# PDF loading and chunk inspection

This notebook walks through the backend's existing PDF-to-chunk pipeline. It calls the same `document_loader` and `split_documents` functions as the chat service. It does not create embeddings, call Gemini, or need an API key.

In VS Code, select the backend project's `.venv` Python interpreter as the notebook kernel before running cells. Then run cells in order. You can adjust `PAGE_INDEX` and `CHUNK_PREVIEW_LENGTH` in later cells to inspect different output.

In [4]:
from pathlib import Path
import statistics
import sys
import time

working_directory = Path.cwd().resolve()
candidate_roots = [working_directory, *working_directory.parents]
candidate_roots.extend(root / "backend" for root in [working_directory, *working_directory.parents])
BACKEND_ROOT = next(
    (root for root in candidate_roots if (root / "app/rag/loader.py").is_file()),
    None,
)
if BACKEND_ROOT is None:
    raise FileNotFoundError("Could not locate backend/app/rag/loader.py from the notebook working directory")

if str(BACKEND_ROOT) not in sys.path:
    sys.path.insert(0, str(BACKEND_ROOT))

DOCUMENTS_DIRECTORY = BACKEND_ROOT / "data" / "documents"
print(f"Notebook working directory: {working_directory}")
print(f"Backend root:              {BACKEND_ROOT}")
print(f"PDF directory:             {DOCUMENTS_DIRECTORY}")

Notebook working directory: /Users/adityakumar/projects/ai_financial_advisor/backend/debug
Backend root:              /Users/adityakumar/projects/ai_financial_advisor/backend
PDF directory:             /Users/adityakumar/projects/ai_financial_advisor/backend/data/documents


## 1. Discover PDF files

The loader searches this directory recursively for files ending in `.pdf`. This cell shows which inputs will be read and their file sizes.

In [5]:
pdf_paths = sorted(DOCUMENTS_DIRECTORY.rglob("*.pdf")) if DOCUMENTS_DIRECTORY.exists() else []
print(f"Found {len(pdf_paths)} PDF file(s).")
for pdf_path in pdf_paths:
    size_mb = pdf_path.stat().st_size / (1024 * 1024)
    print(f"- {pdf_path.relative_to(BACKEND_ROOT)} ({size_mb:.2f} MiB)")

Found 1 PDF file(s).
- data/documents/Data_Scientist_Technical_Interview_Assignment (1)-20260903172457.pdf (0.06 MiB)


## 2. Load PDF pages

`document_loader` uses `PyPDFLoader`. It returns one LangChain `Document` per page, with page text in `page_content` and fields such as the source file and zero-based page number in `metadata`.

In [10]:
from app.rag.loader import document_loader

load_started = time.perf_counter()
documents = document_loader(str(DOCUMENTS_DIRECTORY))
load_seconds = time.perf_counter() - load_started
print(f"Loaded {len(documents)} page document(s) in {load_seconds:.2f} seconds.")

for index, document in enumerate(documents):
    source = Path(str(document.metadata.get("source", "unknown"))).name
    page = document.metadata.get("page")
    page_label = page + 1 if isinstance(page, int) else "unknown"
    text = document.page_content or ""
    preview = " ".join(text.split())[:220]
    print(f"[{index:03}] {source} | page {page_label} | {len(text)} characters")
    print(f"      {preview}")

Loaded 2 page document(s) in 0.15 seconds.
[000] Data_Scientist_Technical_Interview_Assignment (1)-20260903172457.pdf | page 1 | 1842 characters
      DATA SCIENCE / FORECASTING & AGENTIC AI Candidate assignment | Confidential Page 1 of 2 CANDIDATE CHALLENGE Predict & Create Next-Season Styles Use retail data to find the next winners, then turn them into new product id
[001] Data_Scientist_Technical_Interview_Assignment (1)-20260903172457.pdf | page 2 | 1546 characters
      DATA SCIENCE / FORECASTING & AGENTIC AI Candidate assignment | Confidential Page 2 of 2 What we would like you to build 1. Predict the winners Build a model that predicts which styles are most likely to perform strongly 


## 3. Inspect a full extracted page

Change `PAGE_INDEX` to inspect another page. The stored metadata page is zero-based, while the displayed page label is one-based. This output is the text the chunker receives.

In [ ]:
PAGE_INDEX = 0
if not documents:
    print("No pages were loaded. Check that the PDF directory contains readable PDFs.")
elif not 0 <= PAGE_INDEX < len(documents):
    print(f"PAGE_INDEX must be between 0 and {len(documents) - 1}.")
else:
    page_document = documents[PAGE_INDEX]
    page_metadata = page_document.metadata
    print(f"Metadata: {page_metadata}")
    print(f"Extracted text ({len(page_document.page_content)} characters):\n")
    print(page_document.page_content)

## 4. Split pages into chunks

This calls the backend's `split_documents` function. Its `RecursiveCharacterTextSplitter` targets chunks of 1,000 characters with 100 characters of overlap. Actual lengths can vary because the splitter prefers natural text boundaries. Metadata is carried from each page to its chunks.

In [12]:
from app.rag.chunker import split_documents

chunk_started = time.perf_counter()
chunks = split_documents(documents) if documents else []
chunk_seconds = time.perf_counter() - chunk_started
chunk_lengths = [len(chunk.page_content) for chunk in chunks]

print(f"Created {len(chunks)} chunk(s) in {chunk_seconds:.2f} seconds.")
if chunk_lengths:
    print(f"Chunk length characters: min={min(chunk_lengths)}, median={statistics.median(chunk_lengths):.0f}, max={max(chunk_lengths)}")
    print(f"Average chunk length: {statistics.mean(chunk_lengths):.1f}")
else:
    print("No chunks were created.")

Created 4 chunk(s) in 0.00 seconds.
Chunk length characters: min=608, median=970, max=995
Average chunk length: 885.5


## 5. Log every chunk

Each line shows the chunk index, original PDF, human-readable page, and text length. The preview is shortened for readability; set `CHUNK_PREVIEW_LENGTH` higher or print `chunk.page_content` to view a whole chunk.

In [13]:
CHUNK_PREVIEW_LENGTH = 320
for index, chunk in enumerate(chunks):
    source = Path(str(chunk.metadata.get("source", "unknown"))).name
    page = chunk.metadata.get("page")
    page_label = page + 1 if isinstance(page, int) else "unknown"
    content = chunk.page_content or ""
    preview = " ".join(content.split())[:CHUNK_PREVIEW_LENGTH]
    print(f"[{index:03}] {source} | page {page_label} | {len(content)} characters")
    print(f"      {preview}")

[000] Data_Scientist_Technical_Interview_Assignment (1)-20260903172457.pdf | page 1 | 945 characters
      DATA SCIENCE / FORECASTING & AGENTIC AI Candidate assignment | Confidential Page 1 of 2 CANDIDATE CHALLENGE Predict & Create Next-Season Styles Use retail data to find the next winners, then turn them into new product ideas. TIMEBOX SUBMISSION DISCUSSION 6–8 hours Within 5 calendar days Short walkthrough + Q&A The outc
[001] Data_Scientist_Technical_Interview_Assignment (1)-20260903172457.pdf | page 1 | 995 characters
      The new concepts should keep the qualities that may have helped the original styles succeed, while still feeling like genuinely new products rather than copies. The dataset Use the H&M Personalized Fashion Recommendations dataset. It includes customer transactions, detailed article information, and product images linke
[002] Data_Scientist_Technical_Interview_Assignment (1)-20260903172457.pdf | page 2 | 994 characters
      DATA SCIENCE / FORECASTING & AGENTIC 

## 6. Inspect adjacent chunk boundaries

The splitter carries some text from one chunk into the next, helping retrieval retain context near boundaries. This cell compares the previous chunk's ending with the next chunk's beginning and reports their exact shared character prefix/suffix. Boundary-aware splitting means the overlap may be less than the configured target.

In [14]:
BOUNDARY_PREVIEW_LENGTH = 180
for index in range(min(5, max(0, len(chunks) - 1))):
    previous_text = chunks[index].page_content
    next_text = chunks[index + 1].page_content
    max_overlap = min(len(previous_text), len(next_text))
    overlap_length = next((
        size for size in range(max_overlap, 0, -1)
        if previous_text[-size:] == next_text[:size]
    ), 0)
    print(f"Chunks {index} -> {index + 1}: exact shared overlap = {overlap_length} characters")
    print(f"  previous end: {previous_text[-BOUNDARY_PREVIEW_LENGTH:]!r}")
    print(f"  next start:   {next_text[:BOUNDARY_PREVIEW_LENGTH]!r}")

Chunks 0 -> 1: exact shared overlap = 98 characters
  previous end: 'generative AI to create one new product concept inspired by each of those styles.\nThe new concepts should keep the qualities that may have helped the original styles succeed, while'
  next start:   'The new concepts should keep the qualities that may have helped the original styles succeed, while\nstill feeling like genuinely new products rather than copies.\nThe dataset\nUse the'
Chunks 1 -> 2: exact shared overlap = 0 characters
  previous end: ' reliable stock-on-hand or availability history.\nWork with the transactions provided, explain how this limitation affects your prediction, and do\nnot invent missing inventory data.'
  next start:   'DATA SCIENCE / FORECASTING & AGENTIC AI\nCandidate assignment | Confidential Page 2 of 2\nWhat we would like you to build\n1. Predict the winners\nBuild a model that predicts which sty'
Chunks 2 -> 3: exact shared overlap = 56 characters
  previous end: 'P-compatible tools, an

## 7. Pipeline checks

These assertions verify the input and outputs are nonempty and that every chunk retains source metadata. A failure here helps isolate whether the issue is PDF discovery/loading or splitting.

In [15]:
assert pdf_paths, f"No PDFs found under {DOCUMENTS_DIRECTORY}"
assert documents, "PDF files were found, but the loader returned no page documents"
assert chunks, "Page documents were loaded, but the splitter returned no chunks"
assert all(chunk.metadata.get("source") for chunk in chunks), "Some chunks are missing source metadata"

characters_read = sum(len(document.page_content or "") for document in documents)
characters_split = sum(chunk_lengths)
print("Pipeline checks passed.")
print(f"PDF files:       {len(pdf_paths)}")
print(f"Page documents:  {len(documents)}")
print(f"Text chunks:     {len(chunks)}")
print(f"Characters read: {characters_read}")
print(f"Characters split:{characters_split:>8}")
print("No embeddings or network model calls were made.")

Pipeline checks passed.
PDF files:       1
Page documents:  2
Text chunks:     4
Characters read: 3388
Characters split:    3542
No embeddings or network model calls were made.
